In [ ]:
import json
from pathlib import Path

# must import pysr before torch because julia kernel init weirdness
from pysr import PySRRegressor

# torch AFTER pysr, dont let ruff/isort/whatever reorder this
import numpy as np

import sympy
import matplotlib.pyplot as plt
from jaxtyping import Float
from scipy.fft import fft2

In [ ]:
# load activations for each prompt
def load_activations(
	model_name: str,
	base_path: Path = Path("../docs/demo"),
) -> tuple[
	list[dict],
	list[np.lib.npyio.NpzFile],
]:
	model_path: Path = base_path / model_name
	with open(model_path / "prompts.jsonl") as f:
		prompts = [json.loads(line) for line in f]

	activations = [
		np.load(model_path / "prompts" / p["hash"] / "activations.npz") for p in prompts
	]

	return prompts, activations


PROMPTS, ACTIVATIONS = load_activations("pythia-14m")

In [ ]:
ACTIVATIONS[0]

In [ ]:
for i in range(5):
	print(PROMPTS[i]["text"])
	print("=" * 50)

In [ ]:
ACTIVATIONS[1]["blocks.4.attn.hook_pattern"][0, 0].shape

In [ ]:
def plot_pattern_info(
	pattern: Float[np.ndarray, "n_ctx n_ctx"],
	show: bool = True,
) -> tuple[plt.Figure, plt.Axes]:
	gram = pattern @ pattern.T
	fft = fft2(gram)
	fft_shifted = np.fft.fftshift(fft)
	data: dict = {
		"Pattern": pattern,
		"Gram": gram,
		"FFT (abs)": np.abs(fft),
		"FFT (log abs)": np.log(np.abs(fft)),
		"FFT (angle)": np.angle(fft),
		"FFT (real)": np.real(fft),
		"FFT (log abs real)": np.log(np.abs(np.real(fft))),
		"FFT (imag)": np.imag(fft),
		"Shifted FFT (abs)": np.abs(fft_shifted),
		"Shifted FFT (log abs)": np.log(np.abs(fft_shifted)),
		"Shifted FFT (angle)": np.angle(fft_shifted),
		"Shifted FFT (real)": np.real(fft_shifted),
		"Shifted FFT (log abs real)": np.log(np.abs(np.real(fft_shifted))),
		"Shifted FFT (imag)": np.imag(fft_shifted),
	}

	# Create subplots and add colorbars using a loop
	n_data: int = len(data)
	n_cols: int = round(n_data / 2)
	fig, axs = plt.subplots(2, n_cols, figsize=(5 * n_cols, 10))
	for ax, (title, datum) in zip(axs.flat[:n_data], data.items()):
		cax = ax.matshow(datum)
		fig.colorbar(cax, ax=ax)
		ax.set_title(title)
		ax.axis("off")

	if show:
		plt.show()

	return fig, axs


def get_single_attn_pattern(
	sample: int,
	layer: int,
	head: int,
	activations=ACTIVATIONS,
) -> Float[np.ndarray, "n_ctx n_ctx"]:
	return activations[sample][f"blocks.{layer}.attn.hook_pattern"][0, head]


plot_pattern_info(get_single_attn_pattern(1, 4, 0))

In [ ]:
# set up matrix
pattern = get_single_attn_pattern(1, 4, 0)
gram = pattern @ pattern.T
fft = np.fft.fft2(gram)
fft_shifted = np.fft.fftshift(fft)
log_fft = np.log(np.abs(fft_shifted) + 1e-8)  # Add epsilon to prevent log(0)
matrix = log_fft

# set up pysr
model = PySRRegressor(
	# Search parameters
	niterations=1000,
	maxsize=25,
	populations=20,
	population_size=100,
	# Lots of potentially useful operators
	binary_operators=[
		"+",
		"*",
		"-",
		"/",
		"^",
		"abs2(x,y) = x^2 + y^2",  # Radial distance squared
		"hypot(x,y) = sqrt(x^2 + y^2)",  # Radial distance
		# "atan2",  # Angular coordinate
	],
	unary_operators=[
		# Basic functions
		"exp",
		"log1p",
		"abs",
		"sin",
		"cos",
		"tan",
		"sinh",
		"cosh",
		"tanh",
		# Squared terms
		"square(x) = x^2",
		"cube(x) = x^3",
		# Gaussian-like functions
		"gauss(x) = exp(-x^2)",
		"invexp(x) = exp(-abs(x))",
		# Rotated coordinates (various angles)
		# "rot45x(x,y) = x * cos(pi/4) + y * sin(pi/4)",
		# "rot45y(x,y) = -x * sin(pi/4) + y * cos(pi/4)",
		# "rot30x(x,y) = x * cos(pi/6) + y * sin(pi/6)",
		# "rot30y(x,y) = -x * sin(pi/6) + y * cos(pi/6)",
		# "rot60x(x,y) = x * cos(pi/3) + y * sin(pi/3)",
		# "rot60y(x,y) = -x * sin(pi/3) + y * cos(pi/3)",
	],
	# Use batching for speed
	batching=True,
	batch_size=1000,
	# Loss function
	elementwise_loss="loss(prediction, target) = (prediction - target)^2",
	# Define SymPy mappings
	extra_sympy_mappings={
		"square": lambda x: x**2,
		"cube": lambda x: x**3,
		"gauss": lambda x: sympy.exp(-(x**2)),
		"invexp": lambda x: sympy.exp(-abs(x)),
		"abs2": lambda x, y: x**2 + y**2,
		"hypot": lambda x, y: sympy.sqrt(x**2 + y**2),
		# "rot45x": lambda x, y: x * sympy.cos(sympy.pi/4) + y * sympy.sin(sympy.pi/4),
		# "rot45y": lambda x, y: -x * sympy.sin(sympy.pi/4) + y * sympy.cos(sympy.pi/4),
		# "rot30x": lambda x, y: x * sympy.cos(sympy.pi/6) + y * sympy.sin(sympy.pi/6),
		# "rot30y": lambda x, y: -x * sympy.sin(sympy.pi/6) + y * sympy.cos(sympy.pi/6),
		# "rot60x": lambda x, y: x * sympy.cos(sympy.pi/3) + y * sympy.sin(sympy.pi/3),
		# "rot60y": lambda x, y: -x * sympy.sin(sympy.pi/3) + y * sympy.cos(sympy.pi/3),
	},
	# Prevent too much nesting of expensive functions
	nested_constraints={
		"exp": {"exp": 0},
		"gauss": {"gauss": 0},
		"invexp": {"invexp": 0},
	},
	# Other parameters
	parsimony=0.0001,  # Very small to allow complex expressions initially
	turbo=True,  # Speed up evaluation
)
# This will set up the model for 40 iterations of the search code, which contains hundreds of thousands of mutations and equation evaluations.

# Let's train this model on our dataset:
X = np.indices(matrix.shape).reshape(2, -1).T
y = matrix.ravel()
model.fit(X, y)

In [ ]:
model.equations_

In [ ]:
idxs = np.indices(matrix.shape).reshape(2, -1).T
idxs.shape

In [ ]:
def plot_reconstructions(
	original_matrix: np.ndarray, model_equations, start_complexity: int = 2
) -> None:
	"""
	Plot original matrix and reconstructions at each complexity level.

	Args:
	    original_matrix: The original matrix being approximated
	    model_equations: DataFrame containing the PySR equations
	    start_complexity: Minimum complexity to start showing (default 2)
	"""
	# Generate frequency coordinates
	N = original_matrix.shape[0]
	# freqs = np.fft.fftshift(np.fft.fftfreq(N))

	# First, show the original matrix
	plt.figure(figsize=(6, 5))
	plt.imshow(original_matrix, origin="lower")
	plt.colorbar()
	plt.title("Original Matrix")
	plt.xlabel("x")
	plt.ylabel("y")
	plt.show()

	# Get unique complexity levels
	complexities = sorted(model_equations["complexity"].unique())
	complexities = [c for c in complexities if c >= start_complexity]

	# Create coordinate grid
	y, x = np.meshgrid(range(N), range(N))
	coords = np.stack([x.ravel(), y.ravel()], axis=1)

	# For each complexity level
	for complexity in complexities:
		# Get the best equation at this complexity level
		eq_row = model_equations[model_equations["complexity"] == complexity].iloc[-1]
		eq = eq_row["lambda_format"]

		try:
			# Generate prediction
			y_pred = eq(coords)
			reconstructed = y_pred.reshape(original_matrix.shape)

			# Calculate difference
			difference = original_matrix - reconstructed

			# Create comparison plot
			fig, axes = plt.subplots(1, 2, figsize=(12, 5))
			fig.suptitle(
				f'Complexity {complexity} (Loss: {eq_row["loss"]:.4f})\n{eq_row["equation"]}'
			)

			# Reconstructed matrix
			vmin = min(original_matrix.min(), reconstructed.min())
			vmax = max(original_matrix.max(), reconstructed.max())

			im0 = axes[0].imshow(reconstructed, origin="lower", vmin=vmin, vmax=vmax)
			axes[0].set_title("Reconstruction")
			axes[0].set_xlabel("x")
			axes[0].set_ylabel("y")
			fig.colorbar(im0, ax=axes[0])

			# Difference plot
			diff_vmax = max(abs(difference.min()), abs(difference.max()))
			im1 = axes[1].imshow(
				difference,
				origin="lower",
				cmap="RdBu_r",
				vmin=-diff_vmax,
				vmax=diff_vmax,
			)
			axes[1].set_title("Error (Original - Reconstruction)")
			axes[1].set_xlabel("x")
			axes[1].set_ylabel("y")
			fig.colorbar(im1, ax=axes[1])

			plt.tight_layout(rect=[0, 0.03, 1, 0.95])
			plt.show()

		except Exception as e:
			print(f"Error plotting complexity {complexity}: {str(e)}")


# Example usage:
plot_reconstructions(matrix, model.equations_)